In [1]:
import numpy as np
import os
import json

In [ ]:
path = "D:/GAR/data/processed/myogym"
latest_directory = sorted(os.listdir(path))[-1]
print(f"Loading data from: {latest_directory}")
files = os.listdir(os.path.join(path, latest_directory))
print(f"Files in the directory: {files}")

Loading data from: 20260416_131031
Files in the directory: ['metadata.json', 'myogym_test.npz', 'myogym_train.npz', 'myogym_val.npz']


: 

In [ ]:
# load test data
test_data = np.load(os.path.join(path, latest_directory, "myogym_test.npz"), allow_pickle=True)
# load metadata json
with open(os.path.join(path, latest_directory, "metadata.json"), "r") as f:
    metadata = json.load(f)

metadata

{'date': '2026-04-16T13:10:31.933159',
 'dataset': 'myogym',
 'normalization_strategy': None,
 'window_size': 100,
 'window_step': 50,
 'train_ratio': 0.8,
 'val_ratio': 0.2,
 'seed': 42,
 'transform_units': True,
 'units': {'acceleration': 'm/s^2', 'rotation': 'rad/s'},
 'activity_mapping': {'0': 'No activity identified',
  '1': 'Seated Cable Rows',
  '2': 'One-Arm Dumbbell Row',
  '3': 'Wide-Grip Pulldown Behind The Neck',
  '4': 'Bent Over Barbell Row',
  '5': 'Reverse Grip Bent-Over Row',
  '6': 'Wide-Grip Front Pulldown',
  '7': 'Bench Press',
  '8': 'Incline Dumbbell Flyes',
  '9': 'Incline Dumbbell Press',
  '10': 'Dumbbell Flyes',
  '11': 'Pushups',
  '12': 'Leverage Chest Press',
  '13': 'Close-Grip Barbell Bench Press',
  '14': 'Bar Skullcrusher',
  '15': 'Triceps Pushdown',
  '16': 'Bench Dip / Dip',
  '17': 'Overhead Triceps Extension',
  '18': 'Tricep Dumbbell Kickback',
  '19': 'Spider Curl',
  '20': 'Dumbbell Alternate Bicep Curl',
  '21': 'Incline Hammer Curl',
  '22': 

: 

In [ ]:
x_test = test_data["x"]
y_test = test_data["y"]
meta_test = test_data["meta"]
x_test.shape, y_test.shape, meta_test.shape

((7216, 100, 6), (7216,), (7216,))

: 

In [ ]:
from keras import layers, models

: 

In [ ]:
def make_model(T=100, C=6):
    model = models.Sequential([
        layers.Input(shape=(T, C)),

        layers.Conv1D(filters=32, kernel_size=10, padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPool1D(2),
        layers.Dropout(rate=0.25),

        layers.Conv1D(filters=64, kernel_size=10, padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPool1D(2),
        layers.Dropout(rate=0.25),

        layers.Conv1D(filters=64, kernel_size=10, padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPool1D(2),
        layers.Dropout(rate=0.25),

        layers.Conv1D(filters=32, kernel_size=10, padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPool1D(2),
        layers.Dropout(rate=0.25),

        layers.GlobalAveragePooling1D(),
        layers.Dense(1, activation="sigmoid")
    ])
    return model

: 

In [ ]:
model = make_model()
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_8 (Conv1D)               │ (None, 100, 32)        │         1,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 100, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_8 (ReLU)                  │ (None, 100, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_8 (MaxPooling1D)  │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 50, 64)         │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 50, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_9 (ReLU)                  │ (None, 50, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 25, 64)         │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 25, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_10 (ReLU)                 │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 12, 32)         │        20,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 12, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_11 (ReLU)                 │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ (None, 6, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 6, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴─────────────

 Total params: 84,833 (331.38 KB)

 Trainable params: 84,449 (329.88 KB)

 Non-trainable params: 384 (1.50 KB)

: 

In [ ]:
from sklearn.metrics import classification_report

: 

In [ ]:
p_test = model.predict(x_test)
y_test_pred = (p_test >= 0.5).astype(int)
print(classification_report(y_test, y_test_pred, digits=3))


226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
              precision    recall  f1-score   support

           0      0.000     0.000     0.000      5892
           1      0.007     1.000     0.013        47
           2      0.000     0.000     0.000        37
           3      0.000     0.000     0.000        50
           4      0.000     0.000     0.000        32
           5      0.000     0.000     0.000        30
           6      0.000     0.000     0.000        47
           7      0.000     0.000     0.000        27
           8      0.000     0.000     0.000        52
           9      0.000     0.000     0.000        47
          10      0.000     0.000     0.000        60
          11      0.000     0.000     0.000        26
          12      0.000     0.000     0.000        41
          13      0.000     0.000     0.000        35
          14      0.000     0.000     0.000        53
          15      0.000     0.000     0.000        37
          16      0.000     0.000     0.

d:\GAR\envs\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\GAR\envs\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\GAR\envs\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


: 